# Kubernetes Core Objects

A hands-on tour of the core Kubernetes objects on a local k3d cluster (1 server, 3 agents).

- **Cluster basics**: what Kubernetes solves (availability, scalability), control plane vs worker nodes, inspecting clusters and contexts, creating a k3d cluster
- **Namespaces**: listing, creating and switching the default namespace for `kubectl`
- **Workloads**: Pods (CLI and manifest), Deployments (scaling and self-healing), using `describe`
- **Labels**: viewing, selecting (`-l`) and adding labels to resources
- **Networking**: port forwarding, Services (ClusterIP, LoadBalancer) and Endpoints, Ingress with a local hostname (`k8s.local`)
- **Configurations** (ConfigMaps, Secrets)
- **Storage** (PV, PVC, StorageClass), Observability, Security (RBAC)

---

## What is Kubernetes?
(Recap)
- Container orchestration tool (manages containerized applications)
- Manage them in cloud, vm, physical machines or hybrid

## What problems does Kubernetes solve?

Managing applications at scale is HARD. 

- AVAILABILITY - App crashes - Self Healing
- SCALABILITY - Scaling and load balancing 
- DISASTER RECOVER *

## The Cluster Architecture

- Cluster
    - Control Plane (API Server, Controller Manager, Scheduler, ETCD)
    - Worker Nodes (kubelet, kubeproxy)
- Nodes (Runs pods / workloads)

### Get connected clusters

In [ ]:
%%bash
kubectl config get-clusters

In [ ]:
%%bash
kubectl config get-contexts

In [ ]:
%%bash
kubectl top nodes

### Create a cluster

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
if k3d cluster list k8scluster >/dev/null 2>&1; then
  echo "Cluster 'k8scluster' already exists, skipping creation."
else
  k3d cluster create k8scluster \
    --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
    --servers 1 \
    --agents 3
fi

In [ ]:
%%bash
echo ""
kubectl config get-clusters
echo ""
kubectl config get-contexts
echo ""
kubectl top nodes
echo ""
kubectl top pods


## Namespaces  

Resources are ALWAYS creates against a namespace

TYPE is KIND and KIND is TYPE
TYPE is RESOURCE and RESOURCE is TYPE

kubectl get [TYPE] [FLAGS]

In [ ]:
%%bash
kubectl get pods

In [ ]:
%%bash
kubectl get namespaces

In [ ]:
%%bash
# kubectl get pods --all-namespaces
kubectl get pods -A

### Creating a namespace

Not every object in a cluster lives in a namespace.
- StorageClass
- PersistentVolume
- IngressClass

In [ ]:
%%bash
kubectl create namespace k8s-ns

In [ ]:
%%bash
kubectl get namespaces

In [ ]:
%%bash
kubectl config set-context --current --namespace=k8s-ns
kubectl config view --minify | grep namespace:

In [ ]:
%%bash
kubectl get pods

In [ ]:
%%bash
kubectl config set-context --current --namespace=default
kubectl config view --minify | grep namespace:

## Workloads
    - Pods (runs containers)
    - Jobs
    - CronJobs

In [ ]:
%%bash
kubectl get pods -A

### The describe command

kubectl describe [TYPE] [NAME]

In [ ]:
%%bash
# kubectl describe pod [NAME] -n [NAMESPACE]
kubectl describe pod coredns-c4dbffb5f-82mdq -n kube-system

### Pods

#### Using the CLI

In [ ]:
%%bash
kubectl run k8s-pod --image=nginx -n k8s-ns

In [ ]:
%%bash
kubectl get pods -n k8s-ns

In [ ]:
%%bash
kubectl describe pod k8s-pod -n k8s-ns

In [ ]:
%%bash
kubectl delete pod --all -n k8s-ns
kubectl get pods -n k8s-ns

#### Using manifest

In [ ]:
%%bash
#watch on terminal
#watch -n 0.1 kubectl get pods -n k8s-ns
kubectl apply -f manifest/1-create_pod.yml

In [ ]:
%%bash
kubectl delete -f manifest/1-create_pod.yml

### Deployments

#### Using the CLI

In [ ]:
%%bash
kubectl create deployment k8s-deployment --image=nginx -n k8s-ns

In [ ]:
%%bash 
kubectl get deployments -n k8s-ns

##### Deployments are meant to SCALE

In [ ]:
%%bash
kubectl scale deploy k8s-deployment --replicas 3 -n k8s-ns

##### Deployments are meant to be AVAILABLE

In [ ]:
%%bash
# kubectl delete pod [NAME] -n k8s-ns
kubectl delete pod k8s-deployment-df7cc5847-fjpm5 -n k8s-ns

In [ ]:
%%bash
kubectl delete pod --all -n k8s-ns

In [ ]:
%%bash
kubectl delete deployment k8s-deployment -n k8s-ns

#### Using manifest

In [ ]:
%%bash
kubectl apply -f manifest/2-create_deployment.yml

## Labels

The magic clue in a k8s cluster

In [ ]:
%%bash
kubectl get pods --show-labels -n k8s-ns

In [ ]:
%%bash
kubectl get pods -n k8s-ns -l app=k8s-deployment

In [ ]:
%%bash
kubectl get pods,replicaset -n k8s-ns -l app=k8s-deployment

In [ ]:
%%bash
kubectl delete pods,replicaset -n k8s-ns -l app=k8s-deployment

In [ ]:
%%bash
kubectl label pod k8s-deployment-df7cc5847-4w6zl environment=dev -n k8s-ns

In [ ]:
%%bash
kubectl describe pod k8s-deployment-df7cc5847-4w6zl -n k8s-ns

## Networking

> We have workloads that are running in a private virtual network

How can we access the instances of NGINX we created?
- Port Forwarding
- Service & Endpoints
- LoadBalancer
- Ingress & Ingress Controller

In [ ]:
%%bash
kubectl get pods -n k8s-ns -l app=k8s-deployment -o wide

In [ ]:
%%bash

kubectl exec -it k8s-deployment-df7cc5847-7trrm -n k8s-ns  -- sh -c "curl http://localhost"

### Port Forwarding

In [ ]:
%%bash
kubectl port-forward -n k8s-ns deployment/k8s-deployment 8080:80

### Service

Provides a stable IP address to Pods

In [ ]:
%%bash
kubectl get pods -n k8s-ns -l app=k8s-deployment -o wide

In [ ]:
%%bash
kubectl delete pod --all -n k8s-ns

#### Using the CLI

In [ ]:
%%bash
kubectl expose deployment k8s-deployment --port=80 --target-port=8080 --name=k8s-service --type=ClusterIP -n k8s-ns

In [ ]:
%%bash

echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns


In [ ]:
%%bash
kubectl delete pod --all -n k8s-ns

In [ ]:
%%bash

echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns

In [ ]:
%%bash
kubectl delete svc k8s-service -n k8s-ns

k3d cluster edit k8scluster --port-add "8081:8081@loadbalancer"

kubectl expose deployment k8s-deployment \
  --name=k8s-service \
  --type=LoadBalancer \
  --port=8081 \
  --target-port=80 \
  -n k8s-ns

echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns

#### Using manifest

In [ ]:
%%bash
kubectl delete svc k8s-service -n k8s-ns

In [ ]:
%%bash
kubectl apply -f manifest/3-create_service.yml

In [ ]:
%%bash
echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns

In [ ]:
%%bash
kubectl delete -f manifest/3-create_service.yml

### Ingress

In [ ]:
%%bash
kubectl apply -f manifest/4-create_ingress.yml

In [ ]:
%%bash
echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get ingress -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns

In [ ]:
import subprocess
import getpass

hosts = [
    "k8s.local"
]

with open("/etc/hosts", "r") as f:
    current = f.read()

missing = [host for host in hosts if host not in current]
for host in hosts:
    if host not in missing:
        print(f"Already present: {host}")

if missing:
    password = getpass.getpass("sudo password: ")
    for host in missing:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} - {result.stderr.strip()}")

## Configurations
- Config Maps (Volume and Volume Mounts)
- Secrets

### Config Maps

The aim here is to update the default nginx page

In [ ]:
%%bash
# kubectl create cm k8s-cm -n k8s-ns --from-file=files/index.html
kubectl describe cm k8s-cm -n k8s-ns

In [ ]:
%%bash
kubectl apply -f manifest/5-create_volume_mounts.yml
kubectl rollout restart deploy/k8s-deployment -n k8s-ns
kubectl rollout status deploy/k8s-deployment -n k8s-ns


In [ ]:
%%bash
kubectl exec deploy/k8s-deployment -n k8s-ns -- env

In [ ]:
%%bash
kubectl apply -f manifest/6-create_env_configmap.yml
kubectl rollout restart deploy/k8s-deployment -n k8s-ns
kubectl rollout status deploy/k8s-deployment -n k8s-ns

In [ ]:
%%bash
kubectl exec deploy/k8s-deployment -n k8s-ns -- env


## Storage
- Storage Class
- Persistent Volume
- Persistent Volume Claim

In [ ]:
%%bash
kubectl get sc -o wide

### The roles

- **PersistentVolumeClaim (PVC)**: what the app needs (size, access mode). Namespaced.
- **StorageClass**: how to get it (which provisioner, what happens on release, when to bind). Cluster-scoped.
- **PersistentVolume (PV)**: the result, the actual storage. Cluster-scoped.

The app only asks for a claim. The StorageClass decides whether the PV is created for you (dynamic) or has to exist already (static).

In [ ]:
%%bash
kubectl describe sc local-path

### Dynamic vs static provisioning

`manifest/7-create_storage.yml` is file 6 plus:

- **`k8s-pvc-dynamic`**: no StorageClass named, so the default (`local-path`) creates the PV for us
- **`manual` StorageClass** (`kubernetes.io/no-provisioner`, `Retain`) with a **PV we write by hand** and **`k8s-pvc-manual`** that binds to it by class name
- the existing `k8s-deployment` mounts both claims at `/data/dynamic` and `/data/manual`

The deployment drops to **1 replica**: both volumes are `ReadWriteOnce` and node-local, so they cannot be shared across nodes.

In [ ]:
%%bash
kubectl apply -f manifest/7-create_storage.yml
kubectl rollout status deploy/k8s-deployment -n k8s-ns

In [ ]:
%%bash
kubectl get sc -o wide
echo ""
kubectl get pv
echo ""
kubectl get pvc -n k8s-ns
echo ""
kubectl describe pvc k8s-pvc-dynamic -n k8s-ns | grep -A8 Events

#### Copy a file from your machine into the volumes

In [ ]:
%%bash
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl cp files/hello.txt k8s-ns/$POD:/data/dynamic/hello.txt
kubectl cp files/hello.txt k8s-ns/$POD:/data/manual/hello.txt

kubectl exec $POD -n k8s-ns -- cat /data/dynamic/hello.txt
kubectl exec $POD -n k8s-ns -- cat /data/manual/hello.txt

#### The volumes outlive the pod

Delete the pod. The Deployment creates a replacement, and the files are still there.

In [ ]:
%%bash
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl delete pod $POD -n k8s-ns
kubectl rollout status deploy/k8s-deployment -n k8s-ns

POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl exec $POD -n k8s-ns -- cat /data/dynamic/hello.txt
kubectl exec $POD -n k8s-ns -- cat /data/manual/hello.txt

In [ ]:
%%bash
docker exec k3d-k8scluster-agent-0 ls -l /data/k8s-pv-manual

#### Reclaim policy: `Delete` vs `Retain`

A claim in use is protected from deletion, so scale the Deployment to zero first. Then delete both claims:

- `local-path` (`Delete`): the PV and its data are removed
- `manual` (`Retain`): the PV stays in `Released` and the data stays on the node. A released PV is not reused automatically, an administrator must clear its `claimRef` or delete it

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
DYN=$(kubectl get pvc k8s-pvc-dynamic -n k8s-ns -o jsonpath='{.spec.volumeName}')
kubectl scale deploy/k8s-deployment --replicas=0 -n k8s-ns
kubectl wait --for=delete pod -l app=k8s-deployment -n k8s-ns --timeout=60s
kubectl delete pvc k8s-pvc-dynamic k8s-pvc-manual -n k8s-ns
kubectl wait --for=delete pv/$DYN --timeout=60s

echo "== local-path PV is gone, manual PV is Released"
kubectl get pv

echo "== The manual data is still on the node"
docker exec k3d-k8scluster-agent-0 cat /data/k8s-pv-manual/hello.txt

### local-path vs manual

| | `local-path` | `manual` |
|---|---|---|
| Provisioner | `rancher.io/local-path` | `kubernetes.io/no-provisioner` |
| Who creates the PV | Kubernetes, when a pod first uses the claim | You, ahead of time |
| Reclaim policy | `Delete` (PV and data removed with the claim) | `Retain` (PV and data kept) |
| Binding | `WaitForFirstConsumer` | `WaitForFirstConsumer` |
| Good for | Dev and throwaway data | Pre-existing disks, data that must not be lost |

### Clean up storage

Remove the manual PV and class, then re-apply file 6 to put the Deployment back to 3 replicas without volumes.

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
kubectl delete pv k8s-pv-manual --ignore-not-found
kubectl delete sc manual --ignore-not-found
docker exec k3d-k8scluster-agent-0 rm -rf /data/k8s-pv-manual

kubectl apply -f manifest/6-create_env_configmap.yml
kubectl rollout status deploy/k8s-deployment -n k8s-ns

kubectl get sc
kubectl get pv

## Observability

## Security
- Namespaces
- RBAC
- Network Policies

## Cleanup

### Delete Cluster

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# Delete the entire cluster - removes all Docker containers and volumes
k3d cluster delete k8scluster

### Reclaim Disk Space

In [ ]:
%%bash
docker system df
echo ""
echo "=== Dangling volumes ==="
docker volume ls --filter dangling=true
echo ""
docker volume prune -f

### Delete Host Name

In [ ]:
import subprocess
import getpass

hosts = [
    "k8s.local"
]

with open("/etc/hosts", "r") as f:
    lines = f.readlines()

present = [host for host in hosts if any(host in line.split() for line in lines)]
for host in hosts:
    if host not in present:
        print(f"Not present: {host}")

if present:
    password = getpass.getpass("sudo password: ")
    kept = [line for line in lines if not any(host in line.split() for host in present)]
    result = subprocess.run(
        ["sudo", "-S", "tee", "/etc/hosts"],
        input=f"{password}\n{''.join(kept)}",
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        for host in present:
            print(f"Removed: {host}")
    else:
        print(f"Failed: {result.stderr.strip()}")